[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/BradSegal/OxfordADH2026PythonBootcamp/blob/main/extension/solutions/vibe_code_ward_dashboard.ipynb)

In [ ]:
# Setup: run this cell first. It installs the course helpers (about 20 seconds).
import importlib.util, subprocess, sys

if importlib.util.find_spec("adh_python") is None:
    subprocess.run(
        [sys.executable, "-m", "pip", "install", "-q",
         "git+https://github.com/BradSegal/OxfordADH2026PythonBootcamp.git"],
        check=True,
    )
print("Course helpers ready.")

# Extension project: vibe coding a ward early-warning dashboard

**Vibe coding** is the colloquial term for when you describe what you want
to an AI assistant, take the code it produces, and run it (often without
reading it). This can be fast, however, it can also produce code that seems
correct, runs without error, and produces the wrong answer. In a ward, a
wrong early warning score is a patient safety issue.

There isn't a settled science of doing this well as yet, however, what we
can teach is a discipline you already use when doing clinical work which is
to decide what good looks like up front and then check the result against
this. This project has you run this loop once:

1. **Specify.** Write down what the code must do, then turn it into tests.
2. **Generate.** Ask an AI assistant to write the code.
3. **Test and debug.** Run your tests against the code and fix what fails.
4. **Judge.** Compare two AI attempts and decide which you would trust.
5. **Reflect.** Answer a few short questions.

The project is a **ward early warning dashboard** that shows all patients in
a ward ranked by how sick they are according to their NEWS2 score. The
critical element for this from a safety perspective is the function that can
take a single patient's observations and produce a NEWS2 total and risk
band. This is the element you will be required to specify and test.

**You need** Levels 1 to 6. Levels 7 and 8 help but are not required. An AI
assistant helps in Phase 2, but every step works without one.

**Time:** ± 75 minutes.

**One rule before you start.** Never paste real patient information into an
AI assistant unless your organisation has approved that tool for that
information.

In [ ]:
from adh_python import clinical, data
from adh_python.checks import extension as check

ward = data.ward(seed=2026)
admissions = check.new_admissions(first_bed=len(ward) + 1)
ward = ward + admissions

print(len(ward), "patients on the ward. The first one:")
print(ward[0])
print()
print("Admitted from the emergency department tonight:")
for patient in admissions:
    print(
        f"Bed {patient['bed']}: {patient['name']}, {patient['age']}, {patient['diagnosis']}"
    )

## The data

Each patient is a `dict`. Their latest observations are another `dict`
inside of it under the key `"observations"`:

| Key                | Meaning                                                     |
|--------------------|-------------------------------------------------------------|
| `respiration_rate` | breaths per minute                                          |
| `spo2`             | oxygen saturation, %                                        |
| `on_oxygen`        | `True` if the patient is on supplemental oxygen             |
| `systolic_bp`      | systolic blood pressure, mmHg                               |
| `pulse`            | beats per minute                                            |
| `consciousness`    | one ACVPU letter: A alert, C new confusion, V voice, P pain, U unresponsive |
| `temperature`      | °C                                                          |
| `spo2_scale`       | 1 normally; 2 for a patient prescribed a target SpO2 of 88-92% (for example, hypercapnic respiratory failure) |

`data.ward(seed=2026)` is the ward used throughout the course. The three
patients admitted tonight are from the `check.new_admissions`. These are
always the same three patients and you will see each of them again.

## How NEWS2 is calculated (5 minutes)

NEWS2 (the National Early Warning Score 2) turns one round of observations
into a single number that tells the ward team how worried to be. The
calculation has three steps.

**Step 1: score each observation.** Each of the seven parameters scores 0 to
3 from the chart below. A value in the normal range scores 0, and the further
a value is from normal, in either direction, the higher it scores.

| Parameter                  | 3     | 2     | 1         | 0                    | 1               | 2               | 3              |
|----------------------------|-------|-------|-----------|----------------------|-----------------|-----------------|----------------|
| Respiration rate (per min) | ≤8    |       | 9–11      | 12–20                |                 | 21–24           | ≥25            |
| SpO2 scale 1 (%)           | ≤91   | 92–93 | 94–95     | ≥96                  |                 |                 |                |
| SpO2 scale 2 (%)           | ≤83   | 84–85 | 86–87     | 88–92, or ≥93 on air | 93–94 on oxygen | 95–96 on oxygen | ≥97 on oxygen  |
| Air or oxygen              |       | Oxygen |          | Air                  |                 |                 |                |
| Systolic BP (mmHg)         | ≤90   | 91–100 | 101–110  | 111–219              |                 |                 | ≥220           |
| Pulse (per min)            | ≤40   |       | 41–50     | 51–90                | 91–110          | 111–130         | ≥131           |
| Consciousness              |       |       |           | Alert                |                 |                 | New confusion, voice, pain or unresponsive |
| Temperature (°C)           | ≤35.0 |       | 35.1–36.0 | 36.1–38.0            | 38.1–39.0       | ≥39.1           |                |

Every range includes both of its end values: a pulse of exactly 90 scores 0
and a pulse of 91 scores 1. These edges are where scoring code most often
goes wrong.

Three rows need care:

- **SpO2 scale 2** is only for a patient whose clinician has prescribed a
  target saturation of 88–92%, most often because of hypercapnic
  respiratory failure, as in some people with COPD. On this scale 88–92%
  scores 0. Higher saturations still score 0 on air, but score points on
  oxygen, because oxygen pushing them above their target is a risk in
  itself. Use scale 2 only when `spo2_scale` is 2: on anyone else it hides
  low oxygen levels.
- **Air or oxygen** is a parameter of its own. A patient on supplemental
  oxygen scores 2 before anything else is counted.
- **Consciousness** uses ACVPU. Alert scores 0. New confusion (C) scores 3,
  the same as responding only to voice (V) or pain (P), or being
  unresponsive (U).

**Step 2: add up the seven scores.** The total runs from 0 to 20.

**Step 3: turn the total into a risk band.** Check the bands from the top
down and stop at the first that applies:

| Risk band      | When                                      | RCP clinical response               |
|----------------|-------------------------------------------|-------------------------------------|
| `"high"`       | total of 7 or more                        | urgent or emergency response        |
| `"medium"`     | total of 5 or 6                           | key threshold for urgent response   |
| `"low-medium"` | total of 0 to 4, but one parameter scores 3 | urgent ward-based response        |
| `"low"`        | total of 0 to 4, and no parameter scores 3  | ward-based response               |

The order matters. A patient with a total of 8 often has a single 3 as well;
checking `"high"` first means the bigger concern wins, and the single-3 rule
only ever separates low from low-medium.

One thing the chart cannot do is score an observation that was never
taken: there is no row for "not measured". What your code should do then is
part of your specification in Phase 1.

The chart is the Royal College of Physicians NEWS2 (2017). The course
package keeps one tested copy of it in `clinical`, and the checkers use that
copy as their reference. The cell below uses it to score one round of
observations step by step.

In [ ]:
example = {
    "respiration_rate": 22,
    "spo2": 95,
    "on_oxygen": False,
    "systolic_bp": 115,
    "pulse": 105,
    "consciousness": "A",
    "temperature": 38.4,
    "spo2_scale": 1,
}
scores = clinical.news2_parameter_scores(example)
for name, score in scores.items():
    print(f"{name:17} {example[name]!s:>5}   scores {score}")
total = sum(scores.values())
print(f"Total {total}, risk band {clinical.news2_risk(example)!r}")

Two points for the respiration rate and one each for SpO2, pulse and
temperature make a total of 5: medium, an urgent response, although no
single observation looks alarming on its own. That is the point of an
aggregate score.

## Phase 1: Specify before you generate (15 minutes)

If you ask for code before you know what correct looks like, the only way of
judging what is returned is if it looks plausible. Plausible is exactly what
AI code is good at. The specification written first is your answer key.

## Problem E.1: Make, requirements in plain English

Write, in your own words, what a NEWS2 scoring function must do, as if you
were briefing a colleague who will build it. Cover:

- what goes in and what comes out;
- what should happen if an observation is missing; and
- the patients that need special handling: SpO2 scale 2 and new confusion.

Text that runs over several lines is placed between three quote marks at
either end as seen below:

```python
handover = """
Bed 4: new confusion overnight.
Bloods sent, awaiting results.
"""
```

In [ ]:
requirements = """
Input: one round of observations for one patient, as a dict with
respiration_rate, spo2, on_oxygen, systolic_bp, pulse, consciousness
(an ACVPU letter), temperature and spo2_scale (1 or 2).
Output: the NEWS2 total and the risk band (low, low-medium, medium or high),
following the RCP NEWS2 chart (2017).
If any observation is missing, do not guess and do not treat it as normal:
stop with an error that names the missing observation.
Use SpO2 scale 2 only when spo2_scale is 2 (a prescribed target of 88-92%).
New confusion (C) counts the same as V, P or U.
"""
check.requirements(requirements)

Look back at what you wrote. The majority of first drafts describe the
typical patient. Code usually runs into issues with the edge cases, and so
these are the ones most in need of being written down.

## Problem E.2: Modify, turn requirements into test cases

A requirement in English can be read in more than one way. A **test case**
is not, it is one input and the exact output you expect. Here each test
case is a dict with three keys:

- `"name"`: Short description that is displayed when the test runs;
- `"observations"`: the input, a dict like the ones on the ward; and
- `"expected"`: a dict such as `{"total": 3, "risk": "low-medium"}`, or the
  text `"ValueError"` when the code should refuse to give a score. For a
  missing observation, the test runner also checks that the error message
  names it, so whoever reads the message knows what to re-measure.

Risk bands are written exactly as `"low"`, `"low-medium"`, `"medium"` and
`"high"`.

The cell below starts off a list with a full case for a well patient and a
case for you to complete for what **should** happen if the respiration rate
was not recorded. Fill in the `...` in the `expected_when_missing` variable
with your answer. Then add three more cases below where the comment says `#
Your code here`:

1. Value on the edge of a particular NEWS2 band (e.g., the lowest
   respiratory rate that scores a 2 instead of a 0);
2. a patient on SpO2 scale 2;
3. a newly confused patient.

Case 2 records a missing value as `None`, which in Python is the value for
"nothing here". Build each new case similar to case 2, copy the well
patient, modify a property, and then add it to the `my_tests` variable. The
`.copy()` is important, as if you simply wrote `edge = well_patient` you
give the same dictionary a second name, so any change to `edge` changes the
well patient as well.

Work out what the true expected answer is according to the chart. This is
the point, you're writing down what you know to be true! You can use the
reference functions to sanity check individual parameters, e.g.,
`clinical.score_respiration_rate(21)`. If you mess up, the checker compares
the values to the RCP chart and lets you know if you've got something wrong.

**Stretch:** Add a case where the total is directly on the edge of a risk
band.

In [ ]:
well_patient = {
    "respiration_rate": 16,
    "spo2": 97,
    "on_oxygen": False,
    "systolic_bp": 125,
    "pulse": 72,
    "consciousness": "A",
    "temperature": 36.8,
    "spo2_scale": 1,
}

my_tests = [
    {
        "name": "Well patient, every observation normal",
        "observations": well_patient,
        "expected": {"total": 0, "risk": "low"},
    },
]

# Case 2: the respiration rate was not recorded. What should the code do?
no_resp_rate = well_patient.copy()
no_resp_rate["respiration_rate"] = None
expected_when_missing = "ValueError"
my_tests.append(
    {
        "name": "Respiration rate not recorded",
        "observations": no_resp_rate,
        "expected": expected_when_missing,
    }
)

# Case 3: a value exactly on the edge of a NEWS2 band
edge_resp_rate = well_patient.copy()
edge_resp_rate["respiration_rate"] = 21
my_tests.append(
    {
        "name": "Respiration rate 21, the lowest that scores 2",
        "observations": edge_resp_rate,
        "expected": {"total": 2, "risk": "low"},
    }
)

# Case 4: a patient on SpO2 scale 2
scale_2 = well_patient.copy()
scale_2["spo2_scale"] = 2
scale_2["spo2"] = 89
scale_2["on_oxygen"] = True
my_tests.append(
    {
        "name": "Scale 2, SpO2 89% on oxygen, within target",
        "observations": scale_2,
        "expected": {"total": 2, "risk": "low"},
    }
)

# Case 5: a newly confused patient
confused = well_patient.copy()
confused["consciousness"] = "C"
my_tests.append(
    {
        "name": "New confusion, otherwise well",
        "observations": confused,
        "expected": {"total": 3, "risk": "low-medium"},
    }
)

# Stretch: a total exactly on the edge of a risk band
on_the_line = well_patient.copy()
on_the_line["respiration_rate"] = 25
on_the_line["pulse"] = 115
on_the_line["spo2"] = 93
my_tests.append(
    {
        "name": "Total exactly 7",
        "observations": on_the_line,
        "expected": {"total": 7, "risk": "high"},
    }
)

check.test_cases(my_tests)

#### Working out the awkward cases

For example, a scale 2 case with an SpO2 of 89% on oxygen would score 0 for
SpO2 as 89% is within the prescribed target but an additional 2 for being on
oxygen. This would give a total score of 2, low. If the same patient was
scored according to scale 1, they would receive a score of 3 (total of 5,
medium). A case of new confusion in isolation only has a score of 3, but
moves up a risk band to low-medium by virtue of having a 3 in any single
parameter.

### See the test runner work

`check.run_tests` takes your list of tests and any scoring function, runs
all the tests and reports which pass. Note that the function is provided
**as a name and not with brackets** i.e., `check.reference_news2`. This is
because you want to pass the function to the test runner so that it can be
run once per test.

Try it on the course's reference. Every test should pass. If it fails here
it's the test not the code being wrong.

In [ ]:
check.run_tests(my_tests, check.reference_news2)

## Phase 2: Generate (10 minutes)

Now ask an AI assistant to write the scoring function. Use whatever you
have:

- **Gemini in Colab.** If your Colab window shows a Gemini button, it opens
  a chat panel beside your notebook.
- **Another assistant** in a browser tab (ChatGPT / Microsoft Copilot,
  etc.). Use whichever your organisation approves.
- **No assistant?** That is fine. Write the prompt anyway, because writing
  it is half the exercise. Phase 3 gives you code to work with.

### The prompt card

A good request to a coding assistant is a specification not a wish.

Include:

1. **The task in one sentence**, and include the function name you want
   (`score_news2`) so that the code matches your tests.
2. **The input** with a real example of the data structure. Paste in one of
   the patients' observation dicts.
3. **The output** exactly. Here a dict with `"total"` and `"risk"`, and the
   four band labels.
4. **The edge cases** from your requirements and what should happen in each
   e.g., "raise a ValueError that says what observation is missing".
5. **Constraints**: Plain Python, no additional libraries, and readable to a
   beginner.
6. **Ask for tests** with expected answers that you can compare to yours.
7. **Ask it to explain** the code to you in plain English and where it is
   unsure.

And one thing to never include is real patient data.

## Problem E.3: Make, write your prompt

Write your prompt in `my_prompt` according to the prompt card. You can use
the next cell to print an observations dict that you can copy into it. The
checker is a bit of a checklist and is keyword based so if it misses
something because you wrote it in a different way, feel free to proceed.
Copy the text (without the quote marks) to your assistant.

In [ ]:
print(ward[0]["observations"])

In [ ]:
my_prompt = """
Write a Python function score_news2(observations) that calculates the NEWS2
score for one patient, following the Royal College of Physicians NEWS2 chart
(2017).

Input: a dict of one round of observations, like this:
{'respiration_rate': 17, 'spo2': 95, 'on_oxygen': False, 'systolic_bp': 129,
 'pulse': 63, 'consciousness': 'A', 'temperature': 36.9, 'spo2_scale': 1}
consciousness is one ACVPU letter: 'A', 'C', 'V', 'P' or 'U'.
spo2_scale is 1, or 2 for a patient with a prescribed target of 88-92%.

Output: a dict {"total": <whole number>, "risk": <text>}, where risk is one of
"low", "low-medium", "medium" or "high".

Rules:
- If any observation is missing or None, raise a ValueError that names it.
  Never treat a missing observation as normal.
- Use SpO2 scale 2 only when spo2_scale is 2.
- New confusion (C) counts the same as V, P or U.
- Plain Python only, no libraries, written so a beginner can read it.

Then give six test cases (observations and the expected output), including a
missing value, a value on a band edge, a scale 2 patient and a confused
patient. Finally, explain the code in plain English and tell me anything you
are unsure of.
"""
check.prompt(my_prompt)

When the assistant replies, do not run the code it has written yet. Read
through the explanation and test cases it has provided. Do they match your
answers? A disagreement is more useful than an agreement as one of you is
wrong and you need to figure out who.

## Phase 3: Test and debug (15 minutes)

You're going to practise on a draft where we know there's a bug and then try
the same with your own assistant's code.

### How to read a failing test

A failure line indicates which case failed and what was expected versus what
the code gave:

```
❌ Pulse 91: expected total 1, risk low, got total 0, risk low.
```

Then:

1. **Which parameter?** The case changed one thing from the well patient so
   the bug is almost certainly in the code for that one parameter.
2. **Find that part of the code.** Look for its name, here `pulse`.
3. **Compare it to the chart** and decide what it should do.
4. **Change one thing, then re-run every test**, not just the one that
   failed. A fix in one place can break another.

If you cannot see the bug, you can paste the failure line back to the
assistant and ask it to explain the failure before it changes anything. Then
you can re-run all your tests on whatever it gives you.

## Problem E.4: Fix the bug in an assistant's draft

The below is an example of a draft that the assistant would produce for a
prompt similar to the above if informed that the course package has
pre-built, tested functions for calculating the score for each parameter
(`clinical.score_pulse`, etc.). You'll note that it leverages these
functions almost exclusively as in the phase 4 attempts. This has similarly
been written for this course and locked to ensure everyone sees the same
code. The error in this case is a typical error that the assistant would
make.

The cell also shows the draft driving the ward dashboard.

`check.ward_dashboard` provides every patient's observations to whichever
scoring function you hand it and prints the ward sickest first.

Run the cell **before you change anything**. Read the test failures, then
look at Margaret Ellis on the dashboard: she was admitted tonight with new
confusion. Is the risk band shown for her right? Then fix the draft and run
the cell again.

In [ ]:
def draft_score_news2(observations):
    """Return the NEWS2 total and risk band for one round of observations."""
    required = [
        "respiration_rate",
        "spo2",
        "on_oxygen",
        "systolic_bp",
        "pulse",
        "consciousness",
        "temperature",
    ]
    for name in required:
        if observations.get(name) is None:
            raise ValueError(f"Cannot score NEWS2: {name} is missing.")

    if observations.get("spo2_scale", 1) == 2:
        spo2_score = clinical.score_spo2_scale2(
            observations["spo2"], observations["on_oxygen"]
        )
    else:
        spo2_score = clinical.score_spo2_scale1(observations["spo2"])

    consciousness_score = clinical.score_consciousness(observations["consciousness"])

    scores = [
        clinical.score_respiration_rate(observations["respiration_rate"]),
        spo2_score,
        clinical.score_supplemental_oxygen(observations["on_oxygen"]),
        clinical.score_systolic_bp(observations["systolic_bp"]),
        clinical.score_pulse(observations["pulse"]),
        consciousness_score,
        clinical.score_temperature(observations["temperature"]),
    ]
    total = sum(scores)
    if total >= 7:
        risk = "high"
    elif total >= 5:
        risk = "medium"
    elif 3 in scores:
        risk = "low-medium"
    else:
        risk = "low"
    return {"total": total, "risk": risk}


check.run_tests(my_tests, draft_score_news2)
check.ward_dashboard(ward, draft_score_news2)
check.fixed_draft(draft_score_news2)

The draft scores only V, P and U as 3, which is the older NEWS chart
(AVPU). NEWS2 added C for new confusion, scored the same way. Assistants
learn from a great deal of older material, so this is a realistic mistake.
Adding `"C"` to the list fixes it. Calling `clinical.score_consciousness`
fixes it better: notice that the one rule the assistant wrote for itself,
instead of using the tested function, is the one that was wrong.

Before the fix, the dashboard put Margaret Ellis, whose NEWS2 is in the
high band, in the low band, and every other newly confused patient was
under-scored too. Nothing on the screen looked wrong. After the fix, check
that she now appears near the top.

## Problem E.5: Make, test your assistant's code

Paste your assistant's function into the below cell, replacing the
`score_news2 = ...` line. If it named the function something different, just
rename the function to `score_news2`. You can delete any additional example
code it included after the function if you'd like.

**No assistant, or no answer yet?** Write `score_news2 = draft_score_news2`
to use the draft you have just fixed, and carry on.

If something fails, follow the four steps above to resolve or paste the
failure back to the assistant. If it uses Python you haven't seen, ask it to
explain the line to you. After every change, rerun this cell and the next.

In [ ]:
# In this worked version, the fixed draft stands in for an assistant's code.
score_news2 = draft_score_news2
check.run_tests(my_tests, score_news2)

Your tests are only as good as the cases you thought of. The course has a
larger suite, built systematically from the chart: both sides of every band
edge, both SpO2 scales, every ACVPU level, each observation missing in turn
and totals either side of each risk band. Run it on the same code. If your
tests passed but this suite finds failures, your tests had a gap.

Then check the dashboard against what you know. Graham Price, admitted
tonight with COPD, is on SpO2 scale 2: code that ignores the scale puts him
in the high band, when the chart puts him in medium.

In [ ]:
check.run_tests(check.course_tests(), score_news2)
check.ward_dashboard(ward, score_news2)

## Phase 4: Judge two AI attempts (20 minutes)

Here are two attempts that assistants tend to give when given the same
request.

Both run without an error. Please read both and then test them.

Attempt A uses some Python you haven't been exposed to. It uses a
`next(...)` as well as a `for` loop within square brackets `[...]` and
chains `if ... else` statements in a single line. This is part of the
challenge with using an assistant. If it gives you code you can't read, you
won't be able to check if it's correct. You can ask it to explain the code
line-by-line or just trust your tests.

In [ ]:
# Attempt A: compact and table-driven.
def score_news2_a(obs):
    """NEWS2 total and risk band for one round of observations."""
    inf = float("inf")
    spo2_bands = [(91, 3), (93, 2), (95, 1), (inf, 0)]
    if obs.get("spo2_scale") == 2:
        spo2_bands = [(83, 3), (85, 2), (87, 1), (inf, 0)]
        if obs.get("on_oxygen"):
            spo2_bands = [
                (83, 3),
                (85, 2),
                (87, 1),
                (92, 0),
                (94, 1),
                (96, 2),
                (inf, 3),
            ]
    bands = {
        "respiration_rate": [(8, 3), (11, 1), (20, 0), (24, 2), (inf, 3)],
        "spo2": spo2_bands,
        "systolic_bp": [(90, 3), (100, 2), (110, 1), (219, 0), (inf, 3)],
        "pulse": [(40, 3), (50, 1), (90, 0), (110, 1), (130, 2), (inf, 3)],
        "temperature": [(35.0, 3), (36.0, 1), (38.0, 0), (39.0, 1), (inf, 2)],
    }
    scores = [
        next(score for limit, score in table if obs[name] <= limit)
        for name, table in bands.items()
        if obs.get(name) is not None
    ]
    scores.append(2 * bool(obs.get("on_oxygen")))
    scores.append(3 * ((obs.get("consciousness") or "A") != "A"))
    total = sum(scores)
    risk = (
        "high"
        if total > 7
        else "medium"
        if total >= 5
        else "low-medium"
        if 3 in scores
        else "low"
    )
    return {"total": total, "risk": risk}

Attempt B is about four times as long. You do not need to read every
helper function closely; read one of them, then read `score_news2_b` at the
end.

In [ ]:
# Attempt B: one small function per parameter, and input checked first.
NEWS2_OBSERVATIONS_B = [
    "respiration_rate",
    "spo2",
    "on_oxygen",
    "systolic_bp",
    "pulse",
    "consciousness",
    "temperature",
]


def validate_observations_b(observations):
    """Stop with a clear message if an observation is missing or not a valid code."""
    for name in NEWS2_OBSERVATIONS_B:
        if observations.get(name) is None:
            raise ValueError(f"Cannot score NEWS2: {name} is missing.")
    if observations["consciousness"] not in ["A", "C", "V", "P", "U"]:
        raise ValueError(
            "Consciousness must be one of A, C, V, P or U, "
            f"not {observations['consciousness']!r}."
        )
    if observations.get("spo2_scale", 1) not in [1, 2]:
        raise ValueError("spo2_scale must be 1 or 2.")
    if not 0 <= observations["spo2"] <= 100:
        raise ValueError(f"SpO2 must be between 0 and 100, not {observations['spo2']}.")


def respiration_score_b(breaths_per_min):
    """Respiration rate to NEWS2 points."""
    if breaths_per_min <= 8:
        return 3
    if breaths_per_min <= 11:
        return 1
    if breaths_per_min <= 20:
        return 0
    if breaths_per_min <= 24:
        return 2
    return 3


def spo2_score_b(spo2, scale, on_oxygen):
    """Oxygen saturation to NEWS2 points, on scale 1 or scale 2."""
    if scale == 2:
        if spo2 <= 83:
            return 3
        if spo2 <= 85:
            return 2
        if spo2 <= 87:
            return 1
        if spo2 <= 92 or not on_oxygen:
            return 0
        if spo2 <= 94:
            return 1
        if spo2 <= 96:
            return 2
        return 3
    if spo2 <= 91:
        return 3
    if spo2 <= 93:
        return 2
    if spo2 <= 95:
        return 1
    return 0


def systolic_score_b(mmhg):
    """Systolic blood pressure to NEWS2 points."""
    if mmhg <= 90:
        return 3
    if mmhg <= 100:
        return 2
    if mmhg <= 110:
        return 1
    if mmhg <= 219:
        return 0
    return 3


def pulse_score_b(beats_per_min):
    """Pulse to NEWS2 points."""
    if beats_per_min <= 40:
        return 3
    if beats_per_min <= 50:
        return 1
    if beats_per_min <= 90:
        return 0
    if beats_per_min <= 110:
        return 1
    if beats_per_min <= 130:
        return 2
    return 3


def temperature_score_b(celsius):
    """Temperature to NEWS2 points."""
    if celsius <= 35.0:
        return 3
    if celsius <= 36.0:
        return 1
    if celsius <= 38.0:
        return 0
    if celsius <= 39.0:
        return 1
    return 2


def score_news2_b(observations):
    """Return the NEWS2 total and risk band for one round of observations.

    Raises ValueError if an observation is missing, or if consciousness or the
    SpO2 scale is not a valid code, so a gap
    in the chart is never scored as if it were normal.
    """
    validate_observations_b(observations)

    if observations["on_oxygen"]:
        oxygen_score = 2
    else:
        oxygen_score = 0
    if observations["consciousness"] == "A":
        consciousness_score = 0
    else:
        consciousness_score = 3

    scores = [
        respiration_score_b(observations["respiration_rate"]),
        spo2_score_b(
            observations["spo2"],
            observations.get("spo2_scale", 1),
            observations["on_oxygen"],
        ),
        oxygen_score,
        systolic_score_b(observations["systolic_bp"]),
        pulse_score_b(observations["pulse"]),
        consciousness_score,
        temperature_score_b(observations["temperature"]),
    ]
    total = sum(scores)
    if total >= 7:
        risk = "high"
    elif total >= 5:
        risk = "medium"
    elif 3 in scores:
        risk = "low-medium"
    else:
        risk = "low"
    return {"total": total, "risk": risk}

## Problem E.6: Predict

Arjun Mehta was admitted this evening with pneumonia. Let's imagine that his
respiratory rate was not recorded for this round. This is a common issue in
busy wards. The next cell creates a second copy of the ward with this
missing value.

In [ ]:
ward_with_gap = data.ward(seed=2026)
ward_with_gap = ward_with_gap + check.new_admissions(first_bed=len(ward_with_gap) + 1)
for patient in ward_with_gap:
    if patient["name"] == "Arjun Mehta":
        patient["observations"]["respiration_rate"] = None
        gap_observations = patient["observations"]
print(gap_observations)

Read Attempt A again. **Before running anything**, predict what it reports
for Arjun: store the total you think it returns in `prediction`, or the
text `"ValueError"` if you think it will refuse.

In [ ]:
prediction = 4
check.predict_missing_value(score_news2_a, gap_observations, prediction)

Now see what each attempt would place on the dashboard for the ward with the
gap:

In [ ]:
print("Attempt A")
check.ward_dashboard(ward_with_gap, score_news2_a)
print()
print("Attempt B")
check.ward_dashboard(ward_with_gap, score_news2_b)

In attempt A, Arjun is marked in the low band and there is no indication to
anyone on the screen that a value is missing. Attempt B refuses to score him
and places this at the top of the dashboard so that someone will see it.
This is not the software failing, it is the software being honest about what
it doesn't know.

### Run the tests on both attempts

First your own tests then the course suite.

In [ ]:
check.run_tests(my_tests, score_news2_a)
print()
check.run_tests(my_tests, score_news2_b)

In [ ]:
check.run_tests(check.course_tests(), score_news2_a)
print()
check.run_tests(check.course_tests(), score_news2_b)

## Problem E.7: Make, show the second bug

Attempt A has a second bug that has nothing to do with missing values, and
most people's own tests miss it. Build a **complete** set of observations,
nothing missing, on which Attempt A gives the wrong answer, and store it in
`boundary_example`. Then find the line in Attempt A that causes it.

Hint: compare Margaret Ellis and Arjun Mehta on the two dashboards of the
full ward below. The quickest example is one patient's own observations,
such as `admissions[0]["observations"]`. You could also copy
`well_patient` and set the values named in one of the course suite's
failures that is not about a missing value.

In [ ]:
print("Attempt A")
check.ward_dashboard(ward, score_news2_a)
print()
print("Attempt B")
check.ward_dashboard(ward, score_news2_b)

In [ ]:
boundary_example = well_patient.copy()
boundary_example["respiration_rate"] = 25
boundary_example["pulse"] = 115
boundary_example["spo2"] = 93
check.shows_wrong_answer(score_news2_a, boundary_example)

Attempt A has written `total > 7`, the chart indicates that 7 itself is in
the high band. So a total of exactly 7 is coming out as medium. Tonight that
happens to be Margaret Ellis and Arjun Mehta, both of whom are on the cusp
of requiring an emergency response. A single character, `>` instead of `>=`.
No error anywhere, only a test on the exact boundary would find this.

## Weighing the trade-offs

**Readability versus brevity.** Code is read many more times than it is
written: by whoever checks it, whoever fixes it and whoever inherits it.
Attempt A is impressive and hard to check by eye. Attempt B is long, but
each part can be held against the chart line by line.

**Failing loudly versus silently.** A silent wrong answer looks exactly
like a right one. Attempt A's `if obs.get(name) is not None` was probably
meant to be helpful, "skip anything missing", and that is precisely what
makes it dangerous: the missing observation contributes nothing, so a
patient who was not fully observed looks well.

**Hard-coded thresholds versus a single owner.** Both attempts copy the
RCP chart into their own code. If a copy has a typo, or the chart is
revised, every copy must be found and fixed. The draft in Phase 3 used the
course's tested functions instead, so each threshold lives in one place
with its own tests. Its only bug was in the one rule it wrote for itself.

**Who maintains it?** The person who generated the code may move on in four
months. Whoever inherits it needs to understand it, change it safely and
know when it is wrong. Readable code, a test suite and named owners make
that possible; clever code with no tests does not.

## Problem E.8: Decide

Which attempt would you take forward, `"A"` or `"B"`, and why? The
question is not which is perfect: neither is ready for a real ward. Either
choice can be defended, so the checker records your decision rather than
marking it. Store your choice in `choice` and your reasons, a sentence or
two of at least eight words, in `reason`. Base them on what you saw: what
the tests showed, failing loudly or silently, how easily each attempt can
be checked against the chart, and who would maintain it. If you choose A,
say what you would fix first. The checker then comments on your reasons.

In [ ]:
choice = "B"
reason = """
B passed every test, and when an observation was missing it refused and said
which one, so the dashboard showed the gap instead of hiding it. It is long,
but each part can be checked against the chart. Before using it I would
replace its copied thresholds with calls to one tested owner, as the draft did.
"""
check.build_on(choice, reason)

## Before it went near a real ward

Passing tests is where assurance starts, not where it ends. Before a tool
like this is used for real patients, at the very least:

- **Is it needed?** Most trusts already calculate NEWS2 in a validated
  electronic observations system. A second calculator that can disagree
  with the first is a new hazard.
- **Validation.** Compare it with the existing system on properly governed
  data, including late, missing and corrected observations.
- **Clinical safety.** In England, health IT must meet two clinical risk
  management standards: DCB0129 for whoever makes a system and DCB0160 for
  the organisation that deploys and uses it. Both require a named Clinical
  Safety Officer (a registered clinician trained in clinical safety), a
  hazard log and a clinical safety case. A tool built in-house still
  counts: the organisation that builds it is its manufacturer under
  DCB0129, as well as its deployer under DCB0160.
- **Regulation.** Software that calculates a score to guide a patient's
  care may be a medical device, regulated in the UK by the MHRA. Whether it
  is depends on its intended purpose.
- **Information governance.** Real observations are patient data. Where the
  code runs and where the data go, including any AI tool that sees them,
  need approval, for example through a data protection impact assessment.
- **Ownership.** A named team who maintains it, monitors it and can switch
  it off.

## Phase 5: Reflect (5 minutes)

Answer briefly, in your own words:

1. What test cases of yours found a true bug today? What bug did none of
   your own tests catch and what kind of test would have caught it?
2. Attempt A was wrong silently, Attempt B failed loudly. In your own work,
   where would it be worse for an answer to be quietly wrong than to refuse?
   Is there anywhere the opposite is true?
3. If the RCP was to revise one of the bands of the chart tomorrow, what
   would have to change in Attempt A, Attempt B, and the draft?
4. What did the assistant do well today, and what could only you, who
   know the clinical problem, contribute?

Model reflections. Yours will differ, and that is fine.

1. The missing respiration rate case caught Attempt A, and the confused
   patient caught the draft. Only my stretch case, a total exactly on the
   edge of the high band, caught Attempt A's boundary bug. My other cases
   each changed one observation from a well patient, so none of them came
   near that band. The course suite tests every edge systematically, which
   is why it found the bug for people who skipped the stretch.
2. On a ward, a score that quietly ignores a missing observation is worse
   than no score, because it is believed. A refusal costs a nurse a minute
   to re-measure. The opposite can be true where a partial answer is still
   useful and is clearly labelled as partial, for example a dashboard that
   shows "not scored: respiration rate missing" alongside the patients it
   could score.
3. Attempt A: one entry in its bands table. Attempt B: one line in one
   helper function. The draft: nothing, because it calls the tested
   functions in `clinical`, so the one owner changes and its tests prove
   the change. In every case the test suite must be updated first, so it
   can show that the change is right.
4. The assistant was fast, wrote readable code when asked, and explained
   it. It also used an out-of-date chart for confusion, and one attempt
   silently ignored missing data. Only someone who knows the clinical
   problem could say what should happen when an observation is missing,
   which awkward patients matter, and whether the answer was right.

## Before you finish

- Specify first. Your tests are the answer key; write them before you see
  the code.
- A missing observation must never be scored as normal. Prefer code that
  refuses loudly to code that guesses quietly.
- Short code is not safer code. Code you cannot read you can only trust
  through tests.
- Thresholds should be stored within a single tested owner and called.
- Passing tests is the start of clinical safety work not the end.